# **Customer churn**

In [ ]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression 
from sklearn import svm
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn import tree
import seaborn as sns
from sklearn import metrics
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve
)

import matplotlib.pyplot as plt
from IPython.display import display

url = "https://raw.githubusercontent.com/alishermutalov/praktikum-datasets/refs/heads/praktikum/Churn_Modelling.xls"
df = pd.read_csv(url, index_col='RowNumber')
df

# **Analysis**

In [ ]:
# Javobni shu yerda yozing.
print(f"{df.shape}\n")
print(df.isna().sum())


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['Geography'].value_counts()


In [ ]:
#encoder
encoder = LabelEncoder()
df['Geography'] = encoder.fit_transform(df['Geography'])
df.head()

In [ ]:
numeric_df = df.drop(['Surname', 'Gender', 'CustomerId'], axis=1)
numeric_df.corrwith(df['Exited']).abs().sort_values(ascending=False)


In [ ]:
df['Exited'].value_counts() / len(df)*100

In [ ]:
df = df.dropna()

# **Visualisation**

In [ ]:
# Maqsadli o'zgaruvchi taqsimoti
plt.figure(figsize=(8, 5))
sns.countplot(x='Exited', data=df, palette='coolwarm', hue='Exited', legend=False)
plt.title("Mijozlarning Churn taqsimoti", fontsize=14, fontweight='bold')
plt.xlabel("Churn holati (0=Qolgan, 1=Ketgan)", fontsize=12)
plt.ylabel("Mijozlar soni", fontsize=12)
plt.show()

# Churn yuzdagi ma'lumot
exited_counts = df['Exited'].value_counts()
exited_pct = (df['Exited'].value_counts() / len(df) * 100).round(2)
print(f"Bankda qolgan mijozlar: {exited_counts[0]} ({exited_pct[0]}%)")
print(f"Bankdan ketgan mijozlar: {exited_counts[1]} ({exited_pct[1]}%)")

In [ ]:
# Age va Balance ni Exited o'zgaruvchi bilan taqqoslash
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age vs Exited
sns.boxplot(x='Exited', y='Age', data=df, palette='Set2', ax=axes[0])
axes[0].set_title("Yoshni va Churn o'rtasidagi munosabat", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Churn holati (0=Qolgan, 1=Ketgan)", fontsize=11)
axes[0].set_ylabel("Yosh", fontsize=11)

# Balance vs Exited
sns.boxplot(x='Exited', y='Balance', data=df, palette='Set2', ax=axes[1])
axes[1].set_title("Balans va Churn o'rtasidagi munosabat", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Churn holati (0=Qolgan, 1=Ketgan)", fontsize=11)
axes[1].set_ylabel("Balans", fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
df.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20,12))

#Tenure
sns.countplot(x="Tenure", data=df, ax=axes[0,0])
axes[0,0].set_title("Mijoz davomiyligi")

#NumOfProducts
sns.countplot(x="NumOfProducts", data=df, ax=axes[0,1])
axes[0,1].set_title("Mijoz buyurtmalar soni")

#CreditScore
sns.histplot(x="CreditScore", data=df, ax=axes[1,0])
axes[1,0].set_title("Credit ballari")

#EstimatedSalary
sns.histplot(x="EstimatedSalary", data=df, ax=axes[1,1])
axes[1,1].set_title("Yillik maoshi")

# axes[1,1].axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(x='Gender', hue="Exited", palette="viridis", data=df)
plt.title("Qolgan va qaytgan mijozlarning jinsi")
plt.show()

# **Preparation to ML**

In [ ]:
df.drop(["Surname", "Geography"], axis=1, inplace=True)

df['Gender'] = LabelEncoder().fit_transform(df['Gender'])


In [ ]:
x = df.drop("Exited", axis=1)
y = df['Exited']

In [ ]:
x = StandardScaler().fit_transform(x)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

# **ML**

**Logistic Regression**

In [ ]:
lr_model = LogisticRegression()
lr_model.fit(x_train, y_train)
y_pred = lr_model.predict(x_test)

In [ ]:
# testing
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

#Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.title("Logistic Regression Confusion Matrix")
plt.show()

#Roc curve
fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred)
roc_auc = metrics.auc(fpr, tpr)
display = metrics.RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc, estimator_name='ROC curve')
display.plot()
plt.show()


**Support Vector machine**

In [ ]:
svm_model = SVC()
svm_model.fit(x_train, y_train)
y_pred = svm_model.predict(x_test)

# testing
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

#Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.title("SVM Confusion Matrix")
plt.show()

#Roc curve
fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred)
roc_auc = metrics.auc(fpr, tpr)
display = metrics.RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc, estimator_name='ROC curve')
display.plot()
plt.show()

**Decision Tree**

In [ ]:
dec_model = DecisionTreeClassifier(max_depth=4)
dec_model.fit(x_train, y_train)
y_pred = dec_model.predict(x_test)

# testing
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

#Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.title("Decision Tree Confusion Matrix")
plt.show()

#Roc curve
fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred)
roc_auc = metrics.auc(fpr, tpr)
display = metrics.RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc, estimator_name='ROC curve')
display.plot()
plt.show()


**Random Forest**

In [ ]:
ran_model = RandomForestClassifier(n_estimators=7)
ran_model.fit(x_train, y_train)
y_pred = ran_model.predict(x_test)

# testing
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

#Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.title("Random Forest Confusion Matrix")
plt.show()

#Roc curve
fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred)
roc_auc = metrics.auc(fpr, tpr)
display = metrics.RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc, estimator_name='ROC curve')
display.plot()
plt.show()

**XGboost**

In [ ]:
xgb_model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metrics='logloss')
xgb_model.fit(x_train, y_train)
y_pred = xgb_model.predict(x_test)

# testing
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

#Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.title("XGboost Confusion Matrix")
plt.show()

#Roc curve
fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred)
roc_auc = metrics.auc(fpr, tpr)
display = metrics.RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc, estimator_name='ROC curve')
display.plot()
plt.show()

# **Model Comparision**

In [ ]:
from IPython.display import display as ipy_display

models = {
    'Logistic Regression': lr_model,
    'Decision Tree': dec_model,
    'Random Forest': ran_model,
    'SVM': svm_model,
    'XGBoost': xgb_model
}

results = []
for model_name, model in models.items():
    y_pred = model.predict(x_test)
    results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
    })

results_df = pd.DataFrame(results).sort_values(
    by=['F1-Score', 'Accuracy', 'Precision', 'Recall'],
    ascending=False
).reset_index(drop=True)

print('Model comparison table:')
ipy_display(results_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}'
}))

plt.figure(figsize=(10, 5))
melted = results_df.melt(id_vars='Model', value_vars=['Precision', 'Recall', 'F1-Score'])
sns.barplot(data=melted, x='Model', y='value', hue='variable', palette='Set2')
plt.title('Model Metrics Comparison (Precision, Recall, F1-Score)')
plt.xlabel('Model')
plt.ylabel('Score')
plt.ylim(0.0, 1.05)
plt.xticks(rotation=15)
plt.legend(title='Metric')
plt.tight_layout()
plt.show()

best_model = results_df.iloc[0]
print(f"Best model: {best_model['Model']}")